In [1]:
from IPython.display import display
import time
import hmac
import hashlib
import urllib.request
from urllib.parse import quote
import json
import pandas as pd

In [2]:
API_KEY = "9cc2976dc30ff6190b7b0485536cc483"
SECRET  = "61f2cb7a045b539c7a530af70a5be50cbff571f69e6dd8fbaa3dcc1fb8835ac8"

METHOD = "GET"
PATH   = "prediction/playerSeason"
QUERY  = {"p_no": "14170"}

def normalize_query(params: dict) -> str:
    safe = "-_.!~*'()"
    return "&".join(
        f"{quote(str(k), safe=safe)}={quote(str(params[k]), safe=safe)}"
        for k in sorted(params.keys())
    )

def make_signature(secret: str, payload: str) -> str:
    return hmac.new(
        secret.encode("utf-8"),
        payload.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()

normalized = normalize_query(QUERY)
timestamp = str(int(time.time()))

payload = f"{METHOD}|{PATH}|{normalized}|{timestamp}"
signature = make_signature(SECRET, payload)

url = f"https://api.statiz.co.kr/baseballApi/{PATH}?{normalized}"

headers = {
    "X-API-KEY": API_KEY,
    "X-TIMESTAMP": timestamp,
    "X-SIGNATURE": signature,
}

req = urllib.request.Request(url, method=METHOD, headers=headers)

with urllib.request.urlopen(req, timeout=30) as resp:
    body = resp.read().decode("utf-8")

# JSON 변환
json_data = json.loads(body)

# 확인
display(json_data)



{'basic': {'list': [{'p_no': 14170,
    'leagueType': 10100,
    't_code': '6002',
    'p_name': '김대한',
    'year': '2023',
    'p_nameEN': 'Dae-Han Kim',
    'G': '33',
    'WAROff': -0.19222,
    'WARDef': -0.6270180195569992,
    'PA': '89',
    'ePA': '89',
    'AB': '81',
    'R': '10',
    'H': '16',
    '2B': '3',
    '3B': '1',
    'HR': '1',
    'TB': '24',
    'RBI': '7',
    'SB': '1',
    'CS': '3',
    'BB': '7',
    'HP': '1',
    'IB': '0',
    'SO': '21',
    'GDP': '0',
    'SH': '0',
    'SF': '0',
    'AVG': 0.198,
    'OBP': 0.27,
    'SLG': 0.296,
    'OPS': 0.566,
    'wRCplus': 56.91910171508789,
    'WAR': -0.8192383050918579,
    'p_position': 8},
   {'p_no': 14170,
    'leagueType': 10100,
    't_code': '6002',
    'p_name': '김대한',
    'year': '2024',
    'p_nameEN': 'Dae-Han Kim',
    'G': '61',
    'WAROff': -0.659903,
    'WARDef': 0.008163988590240479,
    'PA': '89',
    'ePA': '87',
    'AB': '75',
    'R': '10',
    'H': '10',
    '2B': '1',
    '3B': '

In [10]:
# json 저장
with open("김대한_14170.json", "w", encoding="utf-8") as f:
    json.dump(json_data, f, ensure_ascii=False, indent=4)


In [ ]:
# json 불러오기
with open("김대한_14170.json", "r", encoding="utf-8") as f:
    json_data = json.load(f) # 데이터명 'json_data'

In [3]:
# DataFrame 변환
basic_df = pd.DataFrame(json_data["basic"]["list"])
deepen_df = pd.DataFrame(json_data["deepen"]["list"])
fielding_df = pd.DataFrame(json_data["fielding"]["list"])

display(basic_df.head())
display(deepen_df.head())
display(fielding_df.head())

print(basic_df.columns)

,p_no,leagueType,t_code,p_name,year,p_nameEN,G,WAROff,WARDef,PA,...,GDP,SH,SF,AVG,OBP,SLG,OPS,wRCplus,WAR,p_position
0,14170,10100,6002,김대한,2023,Dae-Han Kim,33,-0.192220,-0.627018,89,...,0,0,0,0.198,0.270,0.296,0.566,56.919102,-0.819238,8
1,14170,10100,6002,김대한,2024,Dae-Han Kim,61,-0.659903,0.008164,89,...,1,2,2,0.133,0.230,0.187,0.417,5.562950,-0.651739,8
2,14170,10100,6002,김대한,2025,Dae-Han Kim,16,-0.184423,-0.138933,37,...,0,0,0,0.194,0.216,0.278,0.494,20.576677,-0.323356,8


,p_no,t_code,p_name,BBK,BABIP,wOBA
0,14170,6002,김대한,0.3333,0.2542,0.274906
1,14170,6002,김대한,0.3077,0.1800,0.213334
2,14170,6002,김대한,0.1667,0.2069,0.223789


,p_no,t_code,p_name,PO,Ass,E,p_position
0,14170,6002,김대한,2,0,0,8
1,14170,6002,김대한,33,1,3,9
2,14170,6002,김대한,16,0,0,7
3,14170,6002,김대한,4,0,0,8
4,14170,6002,김대한,34,0,1,9


Index(['p_no', 'leagueType', 't_code', 'p_name', 'year', 'p_nameEN', 'G',
       'WAROff', 'WARDef', 'PA', 'ePA', 'AB', 'R', 'H', '2B', '3B', 'HR', 'TB',
       'RBI', 'SB', 'CS', 'BB', 'HP', 'IB', 'SO', 'GDP', 'SH', 'SF', 'AVG',
       'OBP', 'SLG', 'OPS', 'wRCplus', 'WAR', 'p_position'],
      dtype='object')


In [5]:
deepen_df["year"] = basic_df["year"].values

display(basic_df)
display(deepen_df)
# fielding 제외

,p_no,leagueType,t_code,p_name,year,p_nameEN,G,WAROff,WARDef,PA,...,GDP,SH,SF,AVG,OBP,SLG,OPS,wRCplus,WAR,p_position
0,14170,10100,6002,김대한,2023,Dae-Han Kim,33,-0.192220,-0.627018,89,...,0,0,0,0.198,0.270,0.296,0.566,56.919102,-0.819238,8
1,14170,10100,6002,김대한,2024,Dae-Han Kim,61,-0.659903,0.008164,89,...,1,2,2,0.133,0.230,0.187,0.417,5.562950,-0.651739,8
2,14170,10100,6002,김대한,2025,Dae-Han Kim,16,-0.184423,-0.138933,37,...,0,0,0,0.194,0.216,0.278,0.494,20.576677,-0.323356,8


,p_no,t_code,p_name,BBK,BABIP,wOBA,year
0,14170,6002,김대한,0.3333,0.2542,0.274906,2023
1,14170,6002,김대한,0.3077,0.1800,0.213334,2024
2,14170,6002,김대한,0.1667,0.2069,0.223789,2025


In [6]:
merged_df = basic_df.merge(
    deepen_df[["year", "BBK", "BABIP", "wOBA"]],
    on="year",
    how="left"
)

display(merged_df.head())
print(merged_df.columns)

,p_no,leagueType,t_code,p_name,year,p_nameEN,G,WAROff,WARDef,PA,...,AVG,OBP,SLG,OPS,wRCplus,WAR,p_position,BBK,BABIP,wOBA
0,14170,10100,6002,김대한,2023,Dae-Han Kim,33,-0.192220,-0.627018,89,...,0.198,0.270,0.296,0.566,56.919102,-0.819238,8,0.3333,0.2542,0.274906
1,14170,10100,6002,김대한,2024,Dae-Han Kim,61,-0.659903,0.008164,89,...,0.133,0.230,0.187,0.417,5.562950,-0.651739,8,0.3077,0.1800,0.213334
2,14170,10100,6002,김대한,2025,Dae-Han Kim,16,-0.184423,-0.138933,37,...,0.194,0.216,0.278,0.494,20.576677,-0.323356,8,0.1667,0.2069,0.223789


Index(['p_no', 'leagueType', 't_code', 'p_name', 'year', 'p_nameEN', 'G',
       'WAROff', 'WARDef', 'PA', 'ePA', 'AB', 'R', 'H', '2B', '3B', 'HR', 'TB',
       'RBI', 'SB', 'CS', 'BB', 'HP', 'IB', 'SO', 'GDP', 'SH', 'SF', 'AVG',
       'OBP', 'SLG', 'OPS', 'wRCplus', 'WAR', 'p_position', 'BBK', 'BABIP',
       'wOBA'],
      dtype='object')


In [7]:
str_cols = ["p_name","p_nameEN","t_code"]
int_cols = [
"p_no","leagueType","year","p_position",
"G","PA","ePA","AB","R","H","2B","3B","HR","TB", # PA(타석), ePA(유효타석), AB(타수), TB(루타수)
"RBI","SB","CS","BB","HP","IB","SO","GDP","SH","SF", # RBI(타점), SB(도루성공), CS(도루자), IB(고의사구), GDP(병살타), SH(희생번트), SF(희생플라이)
]
float_cols = [
"WAROff","WARDef", # 타격WAR, 수비WAR
"AVG","OBP","SLG","OPS","wRCplus","WAR", # OBP(출루율), SLG(장타율)
"BBK","BABIP","wOBA" # BBK(4구/삼진)
]

merged_df[str_cols] = merged_df[str_cols].astype("object")

for c in int_cols:
    if c in merged_df.columns:
        merged_df[c] = pd.to_numeric(merged_df[c], errors="coerce").astype("int64")

for c in float_cols:
    if c in merged_df.columns:
        merged_df[c] = pd.to_numeric(merged_df[c], errors="coerce")

In [8]:
display(merged_df)

merged_df.dtypes

,p_no,leagueType,t_code,p_name,year,p_nameEN,G,WAROff,WARDef,PA,...,AVG,OBP,SLG,OPS,wRCplus,WAR,p_position,BBK,BABIP,wOBA
0,14170,10100,6002,김대한,2023,Dae-Han Kim,33,-0.192220,-0.627018,89,...,0.198,0.270,0.296,0.566,56.919102,-0.819238,8,0.3333,0.2542,0.274906
1,14170,10100,6002,김대한,2024,Dae-Han Kim,61,-0.659903,0.008164,89,...,0.133,0.230,0.187,0.417,5.562950,-0.651739,8,0.3077,0.1800,0.213334
2,14170,10100,6002,김대한,2025,Dae-Han Kim,16,-0.184423,-0.138933,37,...,0.194,0.216,0.278,0.494,20.576677,-0.323356,8,0.1667,0.2069,0.223789


p_no            int64
leagueType      int64
t_code         object
p_name         object
year            int64
p_nameEN       object
G               int64
WAROff        float64
WARDef        float64
PA              int64
ePA             int64
AB              int64
R               int64
H               int64
2B              int64
3B              int64
HR              int64
TB              int64
RBI             int64
SB              int64
CS              int64
BB              int64
HP              int64
IB              int64
SO              int64
GDP             int64
SH              int64
SF              int64
AVG           float64
OBP           float64
SLG           float64
OPS           float64
wRCplus       float64
WAR           float64
p_position      int64
BBK           float64
BABIP         float64
wOBA          float64
dtype: object